# Module 094: Memory Management & Garbage Collection in CPython

Phase 10: Concurrency & Internals | Duration: 2 hours

## Reference Counting

In [ ]:
import sys
from typing import Any, List

obj: List[int] = [1, 2, 3]
print(f"Initial refcount: {sys.getrefcount(obj)}")  # >= 2 (obj + getrefcount arg)

ref: List[int] = obj
print(f"After assignment: {sys.getrefcount(obj)}")

ref = None
print(f"After ref cleared: {sys.getrefcount(obj)}")

del obj
# print(obj)  # NameError: object is deleted

## Circular References

In [ ]:
import gc
from typing import Optional, Any

class Node:
    def __init__(self, name: str) -> None:
        self.name: str = name
        self.next: Optional[Node] = None

    def __del__(self) -> None:
        print(f"__del__ called for {self.name}")

gc.collect()

a = Node("A")
b = Node("B")
a.next = b
b.next = a  # Circular reference

print(f"gc.garbage before delete: {len(gc.garbage)}")
del a
del b

# Reference counting alone cannot free cycles
unreachable: int = gc.collect()
print(f"Unreachable objects collected: {unreachable}")
print(f"gc.garbage after collect: {len(gc.garbage)}")

## Generational GC

In [ ]:
import gc

print(f"GC thresholds: {gc.get_threshold()}")
print(f"GC count: {gc.get_count()}")
print(f"GC stats: {gc.get_stats()}")

print(f"\nGenerations in Python's GC:")
print(f"  Generation 0: youngest objects (collected most frequently)")
print(f"  Generation 1: objects that survived gen 0")
print(f"  Generation 2: oldest objects (collected least frequently)")

## Weak References

In [ ]:
import weakref
import gc
from typing import Any

class Data:
    def __init__(self, value: str) -> None:
        self.value: str = value

obj = Data("important data")
weak: weakref.ref = weakref.ref(obj)

print(f"Weak reference: {weak}")
print(f"Access via weak: {weak().value}")

del obj
gc.collect()
print(f"After delete: {weak()}")  # None

## Memory Diagram

```
CPython Memory Layout:

┌─────────────────────────────────────────────────┐
│  Python Objects (heap)                          │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐       │
│  │ refcnt=3 │ │ refcnt=1 │ │ refcnt=2 │       │
│  │ type=int │ │ type=str │ │ type=list│       │
│  │ val=42   │ │ "hello" │ │ [1,2,3]  │       │
│  └──────────┘ └──────────┘ └──────────┘       │
│                                                  │
│  ┌─────── Generation 0 ───────┐                  │
│  │  Young objects (new ones)  │                  │
│  └────────────────────────────┘                  │
│  ┌─────── Generation 1 ───────┐                  │
│  │  Survivors from gen 0     │                  │
│  └────────────────────────────┘                  │
│  ┌─────── Generation 2 ───────┐                  │
│  │  Oldest objects           │                  │
│  └────────────────────────────┘                  │
└─────────────────────────────────────────────────┘
```